# VGGish

## Model description

VGGish takes audio waveform as input and produces 128-D embedding representations of the semantic content of the waveform. VGGish uses a variant of the VGG architecture and was trained using a preliminary version of the YouTube-8M dataset. VGGish was originally released in the TensorFlow Model Garden, where we have the model source code, the original model checkpoint, and more detailed documentation.

### Inputs

The model accepts a 1-D Tensor or NumPy array containing a waveform of arbitrary length, represented as mono 16 kHz float32 samples in the range [-1.0, +1.0]. Internally, we frame the waveform into sliding windows of 0.96 seconds with no overlap and then run the core of the model on a batch of these frames.

### Outputs

The model returns a 2-D float32 Tensor of shape (N, 128) where N is the number of frames produced as described in the Inputs section above. Each row of the tensor is a 128-D embedding representation of the semantic content of the corresponding frame of audio.

### Preprocessing Steps Needed

The plan is to finetune the model for our speech emotion recognition problem. We have a collection of audio files however befor we begin using the VGGish model we must prepare the data to be within VGGish inputs expecation.

* 16KHz Sampling Rate
* Mono audio
* 0.96 seconds length with no overlap
* compute log-mel spectrograms: [N, 96, 64]

# English Dataset

## Data Preperation

In [6]:
import os
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Official VGGish preprocessing
import vggish_input

DATA_DIR = r"H:\KFUPM AI\ICS504\Project\Data\Organized_Emotions"

X = []
y = []
file_ids = []

data_path = Path(DATA_DIR)

for emotion_folder in data_path.iterdir():
    if not emotion_folder.is_dir():
        continue

    emotion_label = emotion_folder.name

    for wav_file in emotion_folder.glob("*.wav"):
        try:
            # Output shape: [num_patches, 96, 64]
            patches = vggish_input.wavfile_to_examples(str(wav_file))

            for patch in patches:
                X.append(patch)
                y.append(emotion_label)
                file_ids.append(str(wav_file))

        except Exception as e:
            print(f"Error processing {wav_file}: {e}")

X = np.array(X, dtype=np.float32)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("X shape:", X.shape)
print("y shape:", y_encoded.shape)
print("Classes:", label_encoder.classes_)

X shape: (15774, 96, 64)
y shape: (15774,)
Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


In [7]:
np.save("X_vggish.npy", X)
np.save("y_vggish.npy", y_encoded)
np.save("file_ids.npy", np.array(file_ids))

np.save("label_classes.npy", label_encoder.classes_)

In [1]:
import os
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

X = np.load("X_vggish.npy")
y = np.load("y_vggish.npy")
classes = np.load("label_classes.npy", allow_pickle=True)

print(X.shape)
print(y.shape)
print(classes)

(15774, 96, 64)
(15774,)
['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


## Loading Model

In [2]:
import os
import numpy as np

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
tf.reset_default_graph()

import vggish_slim
import vggish_params

checkpoint_path = "vggish_model.ckpt"

X = np.load("X_vggish.npy")

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("X shape:", X.shape)

tf.reset_default_graph()

with tf.Graph().as_default() as graph:
    with tf.Session(graph=graph) as sess:
        vggish_slim.define_vggish_slim(training=True)

        vggish_slim.load_vggish_slim_checkpoint(sess, checkpoint_path)

        features_tensor = graph.get_tensor_by_name(
            vggish_params.INPUT_TENSOR_NAME
        )

        embedding_tensor = graph.get_tensor_by_name(
            vggish_params.OUTPUT_TENSOR_NAME
        )

        embeddings = sess.run(
            embedding_tensor,
            feed_dict={features_tensor: X[:32]}
        )

print("Embeddings shape:", embeddings.shape)

Instructions for updating:
non-resource variables are not supported in the long term
Checkpoint exists: True
X shape: (15774, 96, 64)


H:\conda_envs\rtx_env\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
H:\conda_envs\rtx_env\lib\site-packages\tensorflow\python\keras\legacy_tf_layers\core.py:332: UserWarning: `tf.layers.flatten` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Flatten` instead.
  warnings.warn('`tf.layers.flatten` is deprecated and '


INFO:tensorflow:Restoring parameters from vggish_model.ckpt
Embeddings shape: (32, 128)


### Freeze VGG , Train only the classifier

### Convert VGGish to Keras


In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

# -----------------------------
# Load data
# -----------------------------
X = np.load("X_vggish.npy").astype("float32")
y = np.load("y_vggish.npy")
classes = np.load("label_classes.npy", allow_pickle=True)
file_ids = np.load("file_ids.npy", allow_pickle=True)

num_classes = len(classes)

print("X:", X.shape)
print("y:", y.shape)
print("classes:", classes)
print("num_classes:", num_classes)


# -----------------------------
# Build Keras VGGish backbone
# -----------------------------
def build_keras_vggish_backbone():
    inputs = layers.Input(shape=(96, 64), name="input_features")

    x = layers.Reshape((96, 64, 1), name="reshape")(inputs)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv1")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding="same", name="pool1")(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv2")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding="same", name="pool2")(x)

    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu", name="conv3_1")(x)
    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu", name="conv3_2")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding="same", name="pool3")(x)

    x = layers.Conv2D(512, (3, 3), padding="same", activation="relu", name="conv4_1")(x)
    x = layers.Conv2D(512, (3, 3), padding="same", activation="relu", name="conv4_2")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding="same", name="pool4")(x)

    x = layers.Flatten(name="flatten")(x)

    x = layers.Dense(4096, activation="relu", name="fc1_1")(x)
    x = layers.Dense(4096, activation="relu", name="fc1_2")(x)

    embedding = layers.Dense(128, activation="relu", name="fc2")(x)

    return Model(inputs=inputs, outputs=embedding, name="keras_vggish_backbone")


# -----------------------------
# Build full emotion model
# -----------------------------
backbone = build_keras_vggish_backbone()

inputs = layers.Input(shape=(96, 64), name="emotion_input")

embedding = backbone(inputs)

x = layers.Dropout(0.3, name="emotion_dropout1")(embedding)
x = layers.Dense(64, activation="relu", name="emotion_fc1")(x)
x = layers.Dropout(0.3, name="emotion_dropout2")(x)

outputs = layers.Dense(num_classes, activation="softmax", name="emotion_output")(x)

emotion_model = Model(inputs=inputs, outputs=outputs, name="vggish_emotion_model")


# -----------------------------
# Load pretrained VGGish checkpoint weights
# -----------------------------
layer_name_map = {
    "conv1": "vggish/conv1",
    "conv2": "vggish/conv2",
    "conv3_1": "vggish/conv3/conv3_1",
    "conv3_2": "vggish/conv3/conv3_2",
    "conv4_1": "vggish/conv4/conv4_1",
    "conv4_2": "vggish/conv4/conv4_2",
    "fc1_1": "vggish/fc1/fc1_1",
    "fc1_2": "vggish/fc1/fc1_2",
    "fc2": "vggish/fc2"
}

checkpoint_path = "vggish_model.ckpt"
reader = tf.train.load_checkpoint(checkpoint_path)

for layer in backbone.layers:
    if layer.name in layer_name_map:
        ckpt_name = layer_name_map[layer.name]

        kernel_name = ckpt_name + "/weights"
        bias_name = ckpt_name + "/biases"

        if reader.has_tensor(kernel_name) and reader.has_tensor(bias_name):
            kernel = reader.get_tensor(kernel_name)
            bias = reader.get_tensor(bias_name)

            layer.set_weights([kernel, bias])
            print("Loaded:", layer.name)
        else:
            print("Missing:", layer.name)


# -----------------------------
# Freeze VGGish backbone for Stage 1
# -----------------------------
backbone.trainable = False

emotion_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

emotion_model.summary()

X: (15774, 96, 64)
y: (15774,)
classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
num_classes: 8
Loaded: conv1
Loaded: conv2
Loaded: conv3_1
Loaded: conv3_2
Loaded: conv4_1
Loaded: conv4_2
Loaded: fc1_1
Loaded: fc1_2
Loaded: fc2
Model: "vggish_emotion_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 emotion_input (InputLayer)  [(None, 96, 64)]          0         
                                                                 
 keras_vggish_backbone (Func  (None, 128)              72141184  
 tional)                                                         
                                                                 
 emotion_dropout1 (Dropout)  (None, 128)               0         
                                                                 
 emotion_fc1 (Dense)         (None, 64)                8256      
                                                  

### Split By File

In [5]:


from sklearn.model_selection import train_test_split
import numpy as np

unique_files = np.unique(file_ids)

train_files, test_files = train_test_split(
    unique_files,
    test_size=0.2,
    random_state=42
)

train_idx = np.isin(file_ids, train_files)
test_idx = np.isin(file_ids, test_files)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (12618, 96, 64) (12618,)
Test: (3156, 96, 64) (3156,)


In [6]:
history = emotion_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32
)

Train on 12618 samples, validate on 3156 samples
Epoch 1/10
12618/12618 [==============================] - ETA: 0s - loss: 1.8730 - acc: 0.2733

H:\conda_envs\rtx_env\lib\site-packages\keras\engine\training_v1.py:2332: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


12618/12618 [==============================] - 6s 471us/sample - loss: 1.8730 - acc: 0.2733 - val_loss: 1.6856 - val_acc: 0.3641
Epoch 2/10
12618/12618 [==============================] - 5s 397us/sample - loss: 1.7201 - acc: 0.3359 - val_loss: 1.6072 - val_acc: 0.4043
Epoch 3/10
12618/12618 [==============================] - 5s 397us/sample - loss: 1.6636 - acc: 0.3619 - val_loss: 1.5441 - val_acc: 0.4303
Epoch 4/10
12618/12618 [==============================] - 5s 394us/sample - loss: 1.6338 - acc: 0.3686 - val_loss: 1.5176 - val_acc: 0.4461
Epoch 5/10
12618/12618 [==============================] - 5s 391us/sample - loss: 1.6052 - acc: 0.3882 - val_loss: 1.4812 - val_acc: 0.4585
Epoch 6/10
12618/12618 [==============================] - 5s 395us/sample - loss: 1.5841 - acc: 0.3933 - val_loss: 1.4601 - val_acc: 0.4531
Epoch 7/10
12618/12618 [==============================] - 5s 390us/sample - loss: 1.5747 - acc: 0.3970 - val_loss: 1.4435 - val_acc: 0.4686
Epoch 8/10
12618/12618 [=======

In [7]:
loss, acc = emotion_model.evaluate(X_test, y_test)
print("Test accuracy:", acc)

Test accuracy: 0.48574144


## Finetuning

In [10]:
# -----------------------------
# Correct Stage 2: Actually fine-tune VGGish
# -----------------------------

backbone = emotion_model.get_layer("keras_vggish_backbone")

# Important: parent model must be trainable first
backbone.trainable = True

# Freeze all VGGish layers first
for layer in backbone.layers:
    layer.trainable = False

# Unfreeze only selected top layers
for layer in backbone.layers:
    if layer.name in ["fc1_1", "fc1_2", "fc2"]:
        layer.trainable = True
        print("Unfrozen:", layer.name)

# Recompile after changing trainable flags
emotion_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Trainable layers:")
for layer in backbone.layers:
    if layer.trainable:
        print(layer.name)

print("Total trainable weights:", len(emotion_model.trainable_weights))

Unfrozen: fc1_1
Unfrozen: fc1_2
Unfrozen: fc2
Trainable layers:
fc1_1
fc1_2
fc2
Total trainable weights: 10


In [19]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop_cb = EarlyStopping(
    monitor="val_acc",      # important for your setup
    patience=5,
    restore_best_weights=True,
    verbose=1
)


history_finetune = emotion_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=500,
    batch_size=32,
    callbacks=[early_stop_cb]
)

Train on 12618 samples, validate on 3156 samples
Epoch 1/500
12618/12618 [==============================] - 9s 682us/sample - loss: 0.8633 - acc: 0.6748 - val_loss: 0.9600 - val_acc: 0.6372
Epoch 2/500
12618/12618 [==============================] - 8s 617us/sample - loss: 0.8630 - acc: 0.6769 - val_loss: 0.9593 - val_acc: 0.6372
Epoch 3/500
12618/12618 [==============================] - 8s 628us/sample - loss: 0.8626 - acc: 0.6748 - val_loss: 0.9588 - val_acc: 0.6375
Epoch 4/500
12618/12618 [==============================] - 8s 628us/sample - loss: 0.8569 - acc: 0.6769 - val_loss: 0.9582 - val_acc: 0.6381
Epoch 5/500
12618/12618 [==============================] - 8s 614us/sample - loss: 0.8631 - acc: 0.6743 - val_loss: 0.9575 - val_acc: 0.6381
Epoch 6/500
12618/12618 [==============================] - 8s 611us/sample - loss: 0.8605 - acc: 0.6773 - val_loss: 0.9570 - val_acc: 0.6381
Epoch 7/500
12618/12618 [==============================] - 8s 616us/sample - loss: 0.8558 - acc: 0.6789 -

In [20]:
loss, acc = emotion_model.evaluate(X_test, y_test)
print("Test accuracy:", acc)

Test accuracy: 0.639417


In [21]:
emotion_model.save("stage2_model.h5")

In [8]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    "best_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

In [22]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

patch_probs = emotion_model.predict(X_test, batch_size=32)
test_file_ids = file_ids[test_idx]

file_true = []
file_pred = []

for f in np.unique(test_file_ids):
    idx = test_file_ids == f

    avg_probs = np.mean(patch_probs[idx], axis=0)

    file_pred.append(np.argmax(avg_probs))
    file_true.append(y_test[idx][0])

print("File-level accuracy:", accuracy_score(file_true, file_pred))
print(classification_report(file_true, file_pred, target_names=classes))

H:\conda_envs\rtx_env\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


File-level accuracy: 0.7927756653992395
              precision    recall  f1-score   support

       angry       0.87      0.91      0.89       178
        calm       0.53      0.83      0.65        72
     disgust       0.90      0.81      0.85       125
     fearful       0.72      0.79      0.76       143
       happy       0.81      0.72      0.76       144
     neutral       0.93      0.75      0.83       106
         sad       0.75      0.69      0.72       169
   surprised       0.87      0.85      0.86       115

    accuracy                           0.79      1052
   macro avg       0.80      0.79      0.79      1052
weighted avg       0.81      0.79      0.80      1052



## Results

### Training Accuracy vs Validation Accuracy

### Heat Map

### Classification Report